# Residual Network
作为大二计算机专业、已掌握CNN基础的学生，学习ResNet的核心是**抓住“解决深度网络痛点”的核心思路+关键结构+实践逻辑**，无需陷入复杂公式推导，重点突破以下6个核心模块，就能快速掌握ResNet的精髓：


### 一、先搞懂：ResNet为什么会出现？（核心痛点）
在ResNet之前，人们认为“网络越深，性能越好”，但实际遇到了两个致命问题：
1. **梯度消失/爆炸**（早期痛点）：深层网络反向传播时，梯度会变得极小或极大，导致浅层参数无法更新。  
   ✅ 已部分解决：通过Batch Normalization（BN层）、合理的权重初始化，让梯度能“传递下去”。
2. **退化问题（Degradation）**（ResNet要解决的核心）：当网络深度超过一定阈值（比如20层），**训练误差和测试误差都会上升**（不是过拟合！过拟合是训练误差小、测试误差大）。  
   📌 关键原因：深层网络难以学习“恒等映射”（即输入=输出）——比如给一个已训练好的20层网络加20层“无效层”（输入直接输出），理论上性能应不变，但实际模型学不会这种简单的恒等映射，导致性能下降。

ResNet的核心贡献：**通过“残差连接”，让模型轻松学习恒等映射，从而突破深度限制**（最深做到152层，甚至1000+层）。


### 二、核心思想：残差学习（Residual Learning）
#### 1. 传统CNN的映射逻辑
对于深层网络中的某几层，传统CNN希望这几层直接学习“目标映射” \( H(x) \)（x是输入，H(x)是这几层的理想输出）。

#### 2. ResNet的残差映射逻辑
ResNet不直接学 \( H(x) \)，而是让这几层学习“残差映射” \( F(x) = H(x) - x \)，最终的输出变为：  
\( y = F(x) + x \)  
- 其中 \( F(x) \) 是“残差”（目标输出与输入的差值），由卷积层等学习；  
- \( x \) 是输入，通过“ shortcut connection（ shortcut连接）”直接加到输出端（无参数、无计算量）。

#### 3. 为什么这样更优？
- 若理想映射 \( H(x) = x \)（恒等映射），则只需让 \( F(x) = 0 \)（即残差为0），模型只需把卷积层的权重置为0即可，比直接学恒等映射简单得多；  
- 若理想映射不是恒等映射，模型只需学习“输入到目标输出的差值”，降低了学习难度。


输入 x → [Conv2d(3×3) → BN → ReLU → Conv2d(3×3) → BN] → F(x) → F(x) + x → ReLU → 输出 y
                ↑                                        ↑
                └────────────────  shortcut连接  ─────────┘

<img src = "imgs/ResBlock.png">

- Formally, denoting the desired underlying mapping as H(x), we let the stacked nonlinear layers fit another mapping of F(x) := H(x)−x. The original mapping is recast into F(x)+x. 
- We hypothesize that it is easier to optimize the residual mapping than to optimize the original, unreferenced mapping.

残差输出就是恒等映射加上差值，这样的话如果模型想要学习恒等映射，也就是残差为0的情况，或者类似层输出很小的，更容易学习，因为梯度消失问题在深层神经网络格外显著而梯度极小，具有使得残差接近0的好处

$$
y = \mathcal{F}(x, {W_i}) + x\space\space(1)
$$

其中，$\mathcal{F} = W_2\sigma(W_1x) $, 为了简化标记，$\sigma(W_1x)$表示ReLu激活加上bias
 

- x的维度可能和 $\mathcal{F}(x, W_i)$ 不同， 因此可以线性映射x
$$y = \mathcal{F}(x,{W_i})+W_sx   \space\space(2)$$
也可以采用squre matrix的 $W_i$ 维持维度
- residual 函数$\mathcal{F}$是灵活的，没有固定形式，如果多层layers允许的话，通常有多层layers,如果只有单层layer,等式(1)相当于$y=W_1x+x$，相当一`单层普通的线型层`，失去了残差连接的优势
- 函数 $\mathcal{F}(x,{W_i})$ 可以表示多层conv层，$\mathcal{F}(x,{W_i})$ 和 x 这里是逐元素加法，并且对应channel加上对应channel